# 01 – LiDAR Processing & Structural Metrics

Reproduces Section 3.2.3 and Section 4.1 of the report:
- Load and height-normalise the UAV point cloud
- Compute 23 structural metrics per 20 m pixel (Table 2)
- Write a multiband GeoTIFF used by all downstream notebooks

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root
from src import config
from src.lidar_metrics import (
    load_and_normalise,
    compute_structural_metrics_raster,
    get_bbox_lonlat,
    METRIC_NAMES,
)
import rasterio
import numpy as np
import matplotlib.pyplot as plt

## 1.1 Set the path to your .las file
Edit `config.LAS_FILE` in `src/config.py`, or override here:

In [ ]:
# Override if needed, e.g.:
# config.LAS_FILE = pathlib.Path(r"C:\data\beetaloo_sample.las")

print("LAS file:", config.LAS_FILE)
print("Outputs will go to:", config.OUTPUT_DIR)

## 1.2 Load and height-normalise

In [ ]:
x, y, z_norm, classification, crs = load_and_normalise(config.LAS_FILE)
print(f"Loaded {len(x):,} points")
print(f"Spatial extent: X [{x.min():.1f}, {x.max():.1f}]  Y [{y.min():.1f}, {y.max():.1f}]")
print(f"Height range (after filtering): {z_norm.min():.2f} – {z_norm.max():.2f} m")
print(f"CRS: {crs}")

In [ ]:
z_norm.max()

## 1.3 Height distribution (quick QC)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(z_norm, bins=100, color='forestgreen', edgecolor='none')
axes[0].set(xlabel='Height above ground (m)', ylabel='Count', title='Return height distribution')

axes[1].hist(z_norm[z_norm > 0], bins=100, color='steelblue', edgecolor='none')
axes[1].set(xlabel='Height above ground (m)', ylabel='Count', title='Non-ground returns only')

plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'qc_height_distribution.png', dpi=150)
plt.show()

## 1.4 Compute 23 structural metrics as 20 m resolution raster

In [ ]:
out_path, transform = compute_structural_metrics_raster(x, y, z_norm, classification, crs)
print(f"Written: {out_path}")
print(f"Metrics: {METRIC_NAMES}")

## 1.5 Quick-look: Gap Fraction Probability (GFP), PAI (Plant Area Index), stdev and p50 of canopy height

In [ ]:
with rasterio.open(out_path) as src:
    descriptions = [src.descriptions[i] for i in range(src.count)]
    profile = src.profile

show_metrics = ['GFP', 'PAI', 'stdev', 'p50']
indices = [descriptions.index(m) + 1 for m in show_metrics]

with rasterio.open(out_path) as src:
    data = {m: src.read(i) for m, i in zip(show_metrics, indices)}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
cmaps = ['Greens_r', 'YlGn', 'plasma', 'viridis']
for ax, (m, cmap) in zip(axes, zip(show_metrics, cmaps)):
    arr = data[m]
    arr = np.where(arr == -9999, np.nan, arr)
    im = ax.imshow(arr, cmap=cmap, origin='upper')
    ax.set_title(m)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR /(config.LAS_NAME+ '_qc_structural_metrics.png'), dpi=150)
plt.show()

## 1.6 Save bbox for downstream notebooks

In [ ]:
import json

bbox = get_bbox_lonlat(x, y, crs)
print(f"Bounding box (EPSG:4326): {bbox}")

with open(config.LIDAR_BBOX, 'w') as f:
    json.dump({'minlon': bbox[0], 'minlat': bbox[1], 'maxlon': bbox[2], 'maxlat': bbox[3]}, f, indent=2)
print(f"Saved to {config.LIDAR_BBOX}")